In [ ]:
import torch

print("GPU mövcuddur:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from pathlib import Path
import yaml

dataset_dir = Path(dataset.location).resolve()
data_file = dataset_dir / "data.yaml"

with open(data_file, encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["path"] = str(dataset_dir)
config["train"] = "train/images"
config["val"] = "valid/images"

if (dataset_dir / "test/images").exists():
    config["test"] = "test/images"
else:
    config.pop("test", None)

with open(data_file, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(data_file.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_file),
    epochs=60,
    imgsz=640,
    batch=8,
    patience=15,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    seed=42,
    project="/content/drive/MyDrive/pen_detector/runs",
    name="pen_v1"
)

In [ ]:
from IPython.display import display, Image
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/pen_detector/runs/pen_v1")

display(Image(filename=str(run_dir / "results.png")))

for image_path in sorted(run_dir.glob("val_batch*_pred.jpg")):
    display(Image(filename=str(image_path)))

In [ ]:
from IPython.display import display, Image
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/pen_detector/runs/pen_v1")

for image_path in sorted(run_dir.glob("val_batch*_labels.jpg")):
    display(Image(filename=str(image_path)))


In [ ]:
from ultralytics import YOLO
import torch

model_v2 = YOLO("yolov8n.pt")

results_v2 = model_v2.train(
    data=str(data_file),
    epochs=100,
    patience=35,
    imgsz=640,
    batch=8,

    optimizer="AdamW",
    lr0=0.001,

    mosaic=0.5,
    close_mosaic=10,
    mixup=0.0,

    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    seed=42,

    project="/content/drive/MyDrive/pen_detector/runs",
    name="pen_v2"
)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/pen_detector/runs/pen_v2/weights/best.pt"
)